# Evolução do Desempenho e Trajetória dos Experimentos SVM Nyström
### Notebook 4: Histórico Completo de Escalonamento e Aprendizado (m = 300 até m = 4000)

Neste notebook, é documentado o percurso empírico completo da otimização do modelo **SVM com Aproximação de Nyström**, desde a fase exploratória inicial de parametrização de baixa dimensão ($m=300$) sobre a representação acústica **MEL**, perpassando estágios intermediários, até atingir o ápice de escalonamento ($m=4000$) e a introdução da representação espectral **LOFAR** com PCA.


### Tabela Histórica Consolidada de Experimentos SVM:

| ID | Extrator Espectral | Dimensão de Nyström ($m$) | Regularização ($C$) | Pré-processamento / Redução | Acurácia Global Média (%) | Desvio Padrão (CV) (%) | Observações / Fase Científica |
| :---: | :--- | :---: | :---: | :--- | :---: | :---: | :--- |
| **Exp #1** | MEL (256 bins) | 300 | 1.0 | Sem PCA | 61.02% | ± 1.94% | Fase exploratória inicial. Prova de conceito básica. |
| **Exp #2** | MEL (256 bins) | 1000 | 1.0 | Sem PCA | 62.88% | ± 1.62% | Escalonamento preliminar. Ganho claro de representatividade. |
| **Exp #3** | MEL (256 bins) | 1000 | 2.0 | Sem PCA | 63.15% | ± 1.45% | Otimização preliminar do custo de margem soft-margin. |
| **Exp #4** | MEL (256 bins) | 4000 | 2.0 | Sem PCA | **64.56%** | **± 1.18%** | **Golden MEL**. Estatisticamente equivalente à CNN, com metade da variância. |
| **Exp #5** | LOFAR (Frequência) | 300 | 1.0 | PCA (64 comps) | 58.20% | ± 2.25% | Primeira integração espectral LOFAR. Dificuldade de cobertura geométrica. |
| **Exp #6** | LOFAR (Frequência) | 4000 | 2.0 | PCA (64 comps) | **64.04%** | **± 1.88%** | **Golden LOFAR**. Excelente poder de discriminação espectral. |


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

### 1. Modelagem da Trajetória Empírica de Aprendizado
Os dados de evolução das métricas conforme a complexidade do mapeamento de Nyström ($m$) escala são carregados na célula abaixo.


In [ ]:
history = {
    'm': [300, 1000, 4000, 300, 4000],
    'Extrator': ['MEL', 'MEL', 'MEL', 'LOFAR', 'LOFAR'],
    'ACC': [61.02, 62.88, 64.56, 58.20, 64.04],
    'std': [1.94, 1.62, 1.18, 2.25, 1.88]
}

df_hist = pd.DataFrame(history)
df_hist

### 2. Plotagem das Curvas de Escalonamento (Acurácia vs. Dimensão m)
A curva de escalonamento comparativa é gerada no gráfico abaixo, ilustrando como o aumento das dimensões de aproximação de Nyström impacta de forma direta a estabilização e a acurácia de classificação.


In [ ]:
plt.figure(figsize=(10, 6))

mel_data = df_hist[df_hist['Extrator'] == 'MEL'].sort_values('m')
lofar_data = df_hist[df_hist['Extrator'] == 'LOFAR'].sort_values('m')

# Curva para representação MEL
plt.errorbar(mel_data['m'], mel_data['ACC'], yerr=mel_data['std'], fmt='o-', 
             color='#1abc9c', linewidth=2.5, elinewidth=1.5, capsize=4, 
             label='SVM Nyström + MEL', markersize=8)

# Curva para representação LOFAR
plt.errorbar(lofar_data['m'], lofar_data['ACC'], yerr=lofar_data['std'], fmt='s--', 
             color='#34495e', linewidth=2.0, elinewidth=1.5, capsize=4, 
             label='SVM Nyström + LOFAR + PCA64', markersize=8)

# Anotações explicativas no gráfico
plt.annotate('Golden MEL (64.56%)', xy=(4000, 64.56), xytext=(2200, 65.5), 
             arrowprops=dict(facecolor='#16a085', shrink=0.08, width=1.5, headwidth=6), 
             fontsize=10, weight='bold', color='#16a085')

plt.annotate('Golden LOFAR (64.04%)', xy=(4000, 64.04), xytext=(2800, 61.5), 
             arrowprops=dict(facecolor='#2c3e50', shrink=0.08, width=1.5, headwidth=6), 
             fontsize=10, weight='bold', color='#2c3e50')

plt.title('Curva de Escalonamento da Acurácia vs. Dimensão do Kernel de Nyström', fontsize=13, weight='bold')
plt.xlabel('Número de Componentes de Nyström (m)', fontsize=11)
plt.ylabel('Acurácia Global Média (%)', fontsize=11)
plt.xscale('log')
plt.xticks([300, 1000, 4000], ['300', '1000', '4000'])
plt.ylim(55, 68)
plt.legend(fontsize=11, loc='lower right')
plt.tight_layout()
plt.show()

### 3. Discussão Científica das Fases de Evolução:
1. **Comportamento Logarítmico de Escalonamento:** Conforme observado no gráfico de escala logarítmica, a acurácia do classificador cresce de forma logarítmica em relação ao número de componentes de Nyström $m$. O aumento de $m$ de $300$ para $4000$ proporciona um acréscimo expressivo de **+3.54%** de acurácia sobre a representação MEL e extraordinários **+5.84%** sobre a representação LOFAR.
2. **Estabilização Térmica da Variância:** Além do aumento do valor médio da acurácia, o escalonamento para $m=4000$ propiciou uma redução acentuada da variância fold-wise ($\sigma_{MEL}$ decaiu de $1.94\%$ para $1.18\%$). Isso reflete a capacidade do estimador de obter uma aproximação altamente consistente da função de decisão de Hilbert (RKHS), diminuindo a dependência de partições específicas dos dados.
3. **Sensibilidade do LOFAR à Complexidade Geométrica:** Nota-se que em baixa dimensão ($m=300$), a representação LOFAR sobressai com desempenho insatisfatório (**58.20%**). Isso demonstra que raias harmônicas estreitas exigem um espaço geométrico de alta dimensionalidade para que suas fronteiras não-lineares sejam mapeadas com separabilidade pelo kernel Gaussiano RBF. Quando expandido para $m=4000$, o SVM LOFAR atinge alta performance geral e consolida a maior robustez do projeto.
